# Baseline YOLOv10-S - dataset pills-sxdht

Fine-tuning de yolov10s.pt (preentrenado en COCO) dejando los parámetros por defecto de ultralytics.

Usamos las 7 clases del enunciado (ver la parte de pasar a formato YOLO).

En Colab: Entorno de ejecución -> Cambiar tipo de entorno -> GPU T4

In [ ]:
!pip install -q ultralytics datasets

In [ ]:
from pathlib import Path
import cv2
from datasets import load_dataset
from ultralytics import YOLO
from IPython.display import Image, display

SEED = 0
DATA_DIR = Path("data/pills_yolo")

## Cargar dataset

In [ ]:
ds = load_dataset("Francesco/pills-sxdht")

# nombres de las clases (cambia segun la version de datasets)
objs = ds["train"].features["objects"]
if hasattr(objs, "feature"):
    clases = objs.feature["category"].names  # datasets 3.x
else:
    clases = objs["category"].feature.names  # datasets 4.x (colab)

print(ds)
print(clases)

## Pasar a formato YOLO

Las bbox vienen en formato COCO (x, y, w, h en pixeles) y YOLO quiere un txt por imagen con `clase cx cy w h` normalizado.

El dataset trae 9 clases pero el enunciado dice 7. Quitamos 2:
- `pills`: es la categoria general que mete Roboflow, no tiene ninguna caja.
- `Ibuphil 600 mg`: solo tiene 1 caja en train, 0 en val y 1 en test, no se puede aprender ni validar. Las 2 imagenes donde sale (image_id 12 de train y 61 de test) solo tienen esa pastilla, asi que las quitamos enteras.

Las 7 clases que quedan se renumeran de 0 a 6.

In [ ]:
import shutil

quitar = ["pills", "Ibuphil 600 mg"]
clases_orig = clases
clases = [n for n in clases_orig if n not in quitar]
nuevo_id = {clases_orig.index(n): i for i, n in enumerate(clases)}  # id original -> id nuevo
print(clases)
print(nuevo_id)

shutil.rmtree(DATA_DIR, ignore_errors=True)  # por si habia una conversion anterior

splits = {"train": "train", "validation": "val", "test": "test"}

for split_hf, split_yolo in splits.items():
    (DATA_DIR / "images" / split_yolo).mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "labels" / split_yolo).mkdir(parents=True, exist_ok=True)

    n = 0
    for ej in ds[split_hf]:
        cats = ej["objects"]["category"]
        if any(clases_orig[c] in quitar for c in cats):
            print("  quitamos image_id", ej["image_id"], "de", split_hf)
            continue

        nombre = str(ej["image_id"])
        ej["image"].convert("RGB").save(DATA_DIR / "images" / split_yolo / (nombre + ".jpg"))

        W, H = ej["width"], ej["height"]
        lineas = []
        for (x, y, w, h), c in zip(ej["objects"]["bbox"], cats):
            cx = (x + w / 2) / W
            cy = (y + h / 2) / H
            lineas.append(f"{nuevo_id[c]} {cx} {cy} {w / W} {h / H}")
        with open(DATA_DIR / "labels" / split_yolo / (nombre + ".txt"), "w") as f:
            f.write("\n".join(lineas))
        n += 1

    print(split_hf, n, "imagenes")

# yaml del dataset
with open(DATA_DIR / "data.yaml", "w") as f:
    f.write(f"path: {DATA_DIR.resolve().as_posix()}\n")
    f.write("train: images/train\nval: images/val\ntest: images/test\n")
    f.write("names:\n")
    for i, n in enumerate(clases):
        f.write(f"  {i}: '{n}'\n")

print(open(DATA_DIR / "data.yaml").read())

## Entrenamiento

Todo por defecto (100 epocas, imgsz 640, batch 16, optimizer auto...). Solo fijamos la semilla. Usa val para quedarse con el mejor modelo (best.pt).

In [ ]:
model = YOLO("yolov10s.pt")
model.train(data=str(DATA_DIR / "data.yaml"), seed=SEED, name="yolov10s_baseline")

carpeta = Path(model.trainer.save_dir)  # runs/detect/yolov10s_baseline...
print(carpeta)

In [ ]:
# curvas de loss y mAP en val
display(Image(filename=carpeta / "results.png", width=900))

## Evaluacion en test

In [ ]:
best = YOLO(carpeta / "weights" / "best.pt")
metrics = best.val(data=str(DATA_DIR / "data.yaml"), split="test", name="yolov10s_baseline_test")

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("precision:", metrics.box.mp, "recall:", metrics.box.mr)
print("tiempos (ms por imagen):", metrics.speed)

In [ ]:
# AP por clase
for i, c in enumerate(metrics.box.ap_class_index):
    print(clases[c], "AP50 =", round(metrics.box.ap50[i], 3), " AP50-95 =", round(metrics.box.ap[i], 3))

In [ ]:
# curva PR y matriz de confusion (las guarda ultralytics)
display(Image(filename=Path(metrics.save_dir) / "BoxPR_curve.png", width=700))
display(Image(filename=Path(metrics.save_dir) / "confusion_matrix.png", width=700))

In [ ]:
# algunas predicciones en test
imgs = sorted((DATA_DIR / "images" / "test").glob("*.jpg"))[:6]
for r in best.predict(imgs, verbose=False):
    display(Image(data=cv2.imencode(".jpg", r.plot())[1].tobytes(), width=400))